<a href="https://colab.research.google.com/github/mahamrija/MahamRija-AnalyticsEngineering-/blob/main/2_03_practice_bike_hire_student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![](https://drive.google.com/uc?export=view&id=17M9oGaOj4KdKiDDH3oTibxJ9qBTSjp6n)

# 2.03 Practice session
## Bike-hire trips

## How this practice works

This is a **short rehearsal** of the timed test. You do not need to finish or get the "right" answer. The aim is to practise the way of working.

- You have about **45 minutes**. Work individually.
- You may use **any AI tool** (for example Gemini in Colab). We are **not marking** anything today. In the real test, the quality of AI-generated code will not matter. What matters is the control youDshow.
- Write in markdown cells as you go: your thinking, **each prompt you send (paste it as you send it)**, and the checks you make.
- Unfinished work is fine. Say what you did not get to.
- At the end, download the Notebook (using File > Download > Download .ipynb). Once it has completed please zip the file up. This is the artefact you would submit for the in-class test.
- You can also save a copy to your GitHub using the instructions from my.wbs (1b asychronous tasks).

**Suggested timing**

| Minutes | Focus |
| --- | --- |
| 0 to 10 | Look at the tables and note what you find |
| 10 to 25 | Clean and prepare the data |
| 25 to 38 | Join and aggregate to answer the question |
| 38 to 45 | Check your answer and write up |

At the end, use **Runtime, then Restart session and run all** and note anything that fails.


## The data

A fictional bike-hire scheme: three tables covering trips taken in March 2025.

| Table | What it holds |
| --- | --- |
| `trips` | One record per trip: bike, start and end station, start time, duration and user type |
| `bikes` | The bikes, with type and purchase year |
| `stations` | The docking stations, with zone and capacity |

**How they connect:** `trips.bike_id` to `bikes.bike_id`, and `trips.start_station_id` (and `end_station_id`) to `stations.station_id`.

Check the real column names in the next cell before you write anything.


In [1]:
# ---- Setup: run this cell first ----
!pip install -q duckdb

import glob
import os

import duckdb
import pandas as pd

DATA_URL = "https://github.com/MJMortensonWarwick/adminaeaiv/releases/download/dbsession/bike-hire-practice.zip"

DATA_DIR = "/content/bikes"
# Download only if no CSV files are there yet. The CSVs may sit inside a folder
# within the zip; the search below looks in sub-folders too.
if DATA_URL and not glob.glob(os.path.join(DATA_DIR, "**", "*.csv"), recursive=True):
    import urllib.request
    import zipfile
    os.makedirs(DATA_DIR, exist_ok=True)
    urllib.request.urlretrieve(DATA_URL, "/content/bikes.zip")
    zipfile.ZipFile("/content/bikes.zip").extractall(DATA_DIR)

FILES = {"trips": "trips.csv", "bikes": "bikes.csv", "stations": "stations.csv"}

tables, missing = {}, []
for name, filename in FILES.items():
    found = glob.glob(os.path.join(DATA_DIR, "**", filename), recursive=True)
    if found:
        tables[name] = pd.read_csv(found[0])
    else:
        missing.append(filename)

if missing:
    found_csvs = sorted(os.path.basename(f) for f in glob.glob(os.path.join(DATA_DIR, "**", "*.csv"), recursive=True))
    raise FileNotFoundError(
        f"Missing files: {missing}. CSV files found under {DATA_DIR}: {found_csvs}. Ask for help."
    )

# Make every table available as a DuckDB table, so you can use SQL or pandas.
con = duckdb.connect()
for name, df in tables.items():
    con.register(name, df)


def q(sql):
    """Run SQL against the tables above and return a pandas DataFrame."""
    return con.sql(sql).df()


for name, df in tables.items():
    print(f"{name}: {df.shape[0]:,} rows, {df.shape[1]} columns")
    print("   ", list(df.columns))


trips: 612 rows, 7 columns
    ['trip_id', 'bike_id', 'start_station_id', 'end_station_id', 'start_time', 'duration_min', 'user_type']
bikes: 60 rows, 3 columns
    ['bike_id', 'bike_type', 'purchase_year']
stations: 12 rows, 4 columns
    ['station_id', 'station_name', 'zone', 'capacity']


## Examples: two ways to work with the tables

You can use SQL, pandas, or a mix of both. The next two cells show each. Run them and change them to see what happens. You do not have to use them in your answer.


In [2]:
# Example 1: SQL
# q() runs SQL against any of the tables above and returns a pandas DataFrame.
from IPython.display import display

display(q("SELECT * FROM trips LIMIT 5"))

display(q("SELECT COUNT(*) AS n_rows, COUNT(DISTINCT trip_id) AS n_distinct_trips FROM trips"))

display(q("""
    SELECT t.trip_id, s.station_name
    FROM trips AS t
    JOIN stations AS s ON t.start_station_id = s.station_id
    LIMIT 5
"""))


,trip_id,bike_id,start_station_id,end_station_id,start_time,duration_min,user_type
0,T0001,B028,S05,S01,2025-03-01 05:52:00,21.0,Casual
1,T0002,B043,S12,S12,2025-03-01 06:31:00,27.0,CASUAL
2,T0003,B045,S11,S07,2025-03-01 07:06:00,18.0,Casual
3,T0004,B051,S04,S04,2025-03-01 07:06:00,34.0,Member
4,T0005,B049,S01,S09,2025-03-01 07:36:00,16.0,Member


,n_rows,n_distinct_trips
0,612,600


,trip_id,station_name
0,T0001,Park Road
1,T0002,Cathedral Close
2,T0003,Sports Centre
3,T0004,University Gate
4,T0005,Market Square


In [4]:
# Example 2: pandas
# Each table is also a pandas DataFrame, stored in the dictionary `tables`.
from IPython.display import display

trips = tables["trips"]
display(trips.head())
print(trips.shape)
print(trips.dtypes)

joined = tables["trips"].merge(tables["stations"], left_on="start_station_id", right_on="station_id", how="inner")
display(joined[["trip_id", "station_name"]].head())

# You can mix the two: run SQL, then use pandas methods on the result
result = q("SELECT start_station_id FROM trips")
print(result["start_station_id"].nunique(), "different start stations in trips")


,trip_id,bike_id,start_station_id,end_station_id,start_time,duration_min,user_type
0,T0001,B028,S05,S01,2025-03-01 05:52:00,21.0,Casual
1,T0002,B043,S12,S12,2025-03-01 06:31:00,27.0,CASUAL
2,T0003,B045,S11,S07,2025-03-01 07:06:00,18.0,Casual
3,T0004,B051,S04,S04,2025-03-01 07:06:00,34.0,Member
4,T0005,B049,S01,S09,2025-03-01 07:36:00,16.0,Member


(612, 7)
trip_id              object
bike_id              object
start_station_id     object
end_station_id       object
start_time           object
duration_min        float64
user_type            object
dtype: object


,trip_id,station_name
0,T0001,Park Road
1,T0002,Cathedral Close
2,T0003,Sports Centre
3,T0004,University Gate
4,T0005,Market Square


12 different start stations in trips


## Your brief

**From:** the scheme manager

> We want to know whether electric bikes are used for longer trips than standard bikes, and whether that differs by zone. Please give us the **average trip duration by bike type and by the zone of the start station**.

![](https://drive.google.com/uc?export=view&id=1eH5Q3l4LnzwZ-XcT_kJDeHoKA2rufiOW)

**What your final table should look like**

One row per combination of bike type and zone, with the number of trips and the average duration in minutes. State what one row represents.

**Things to think about** (you do not have to solve all of them)

- Are there missing, duplicated or odd values in the tables you use? What will you do about each, and why?
- Do the bike types and dates look consistent?
- How will you check your final table is believable?


## 1. Look at the data

*What does one row of each table represent? Which columns will you use? What looks odd, missing or inconsistent? Write down what you find.*

In [ ]:
# Check missing values in all three tables
for name, df in tables.items():
    print(f"\n{name}")
    print(df.isnull().sum())


trips
trip_id              0
bike_id              0
start_station_id     0
end_station_id       0
start_time           0
duration_min        30
user_type            0
dtype: int64

bikes
bike_id          0
bike_type        0
purchase_year    0
dtype: int64

stations
station_id      0
station_name    0
zone            0
capacity        1
dtype: int64


In [ ]:

# Display all rows with repeated trip IDs
duplicate_trips = trips[
    trips["trip_id"].duplicated(keep=False)
].sort_values("trip_id")

display(duplicate_trips)


,trip_id,bike_id,start_station_id,end_station_id,start_time,duration_min,user_type
96,T0097,B031,S12,S12,2025-03-06 08:14:00,10.0,Casual
97,T0097,B031,S12,S12,2025-03-06 08:14:00,10.0,Casual
175,T0175,B059,S04,S09,2025-03-09 21:51:00,13.0,Casual
176,T0175,B059,S04,S09,2025-03-09 21:51:00,13.0,Casual
185,T0184,B034,S02,S03,2025-03-10 12:23:00,25.0,Casual
186,T0184,B034,S02,S03,2025-03-10 12:23:00,25.0,Casual
285,T0283,B014,S07,S04,2025-03-15 19:48:00,5.0,Member
286,T0283,B014,S07,S04,2025-03-15 19:48:00,5.0,Member
326,T0323,B023,S08,S05,2025-03-17 17:17:00,52.0,Member
327,T0323,B023,S08,S05,2025-03-17 17:17:00,52.0,Member


In [ ]:

# Summarise trip durations
print(trips["duration_min"].describe())

# Display zero or negative durations
display(trips[trips["duration_min"] <= 0])


count     582.000000
mean       27.891753
std       145.064305
min         4.000000
25%        12.000000
50%        16.000000
75%        22.000000
max      2880.000000
Name: duration_min, dtype: float64


,trip_id,bike_id,start_station_id,end_station_id,start_time,duration_min,user_type




**What each row represents and columns I will use**

- **Trips:** One row should represent one trip. I will use `trip_id`, `bike_id`, `start_station_id`, `start_time` and `duration_min`.
- **Bikes:** One row represents one bike. I will use `bike_id` and `bike_type`.
- **Stations:** One row represents one station. I will use `station_id` and `zone`.

**What looks odd, missing or inconsistent**

The trips table has 612 rows but only 600 distinct trip IDs, suggesting 12 extra rows with repeated IDs. I need to investigate these before removing anything. The `start_time` column is stored as text rather than a datetime, and `user_type` has inconsistent capitalisation, such as `Casual` and `CASUAL`. I also need to check for missing values, invalid trip durations, and inconsistencies in bike types or station IDs.

I will use the bike type and the zone of the starting station to group the trips and calculate the number of trips and average duration in minutes.

## 2. Clean and prepare

*Decide what to do about each problem you found, say why, and do it. Short cells that you can re-run are better than one long cell.*

In [8]:

# Keep one copy of each completely identical trip record
trips_clean = trips.drop_duplicates().copy()

print("Rows before:", len(trips))
print("Rows after:", len(trips_clean))
print("Distinct trip IDs:", trips_clean["trip_id"].nunique())


Rows before: 612
Rows after: 600
Distinct trip IDs: 600


In [9]:

# Convert mixed date formats into a consistent datetime format
trips_clean["start_time"] = pd.to_datetime(
    trips_clean["start_time"],
    format="mixed",
    dayfirst=True,
    errors="coerce"
)

print("Missing or unparsed start times:",
      trips_clean["start_time"].isna().sum())


Missing or unparsed start times: 0


In [10]:

# Display the longest trips for inspection
display(
    trips_clean.nlargest(10, "duration_min")[
        ["trip_id", "start_time", "duration_min", "bike_id"]
    ]
)


,trip_id,start_time,duration_min,bike_id
80,T0081,2025-03-04 21:57:00,2880.0,B010
143,T0143,2025-03-08 12:23:00,1440.0,B020
287,T0284,2025-03-15 20:25:00,1440.0,B046
127,T0127,2025-03-07 09:01:00,90.0,B025
357,T0352,2025-03-19 08:59:00,67.0,B042
326,T0323,2025-03-17 17:17:00,52.0,B023
520,T0512,2025-03-26 18:32:00,51.0,B045
37,T0038,2025-03-02 12:23:00,50.0,B016
262,T0260,2025-03-15 07:29:00,46.0,B025
268,T0266,2025-03-15 13:53:00,46.0,B046


In [12]:

# Check missing values in the cleaned trips table
print(trips_clean.isnull().sum())

# Check missing values in the bikes and stations tables
print("\nBikes:")
print(tables["bikes"].isnull().sum())

print("\nStations:")
print(tables["stations"].isnull().sum())


trip_id              0
bike_id              0
start_station_id     0
end_station_id       0
start_time           0
duration_min        30
user_type            0
dtype: int64

Bikes:
bike_id          0
bike_type        0
purchase_year    0
dtype: int64

Stations:
station_id      0
station_name    0
zone            0
capacity        1
dtype: int64




I will remove exact duplicate trip records because they would inflate the trip count and could affect the average duration. I will standardise the `start_time` format to datetime and check missing values in the columns needed for the analysis. I will investigate the unusually long 2,880-minute trip before deciding whether to exclude it. I will also standardise text capitalisation where needed. These steps should make the final grouping more reliable.
The longest trips include one lasting 2,880 minutes and two lasting 1,440 minutes. These are potential outliers compared with the median duration of 16 minutes. I will flag them for investigation rather than automatically remove them, because the available data does not confirm that they are errors. Exact duplicate rows will be removed to avoid double-counting trips.

The missing-value check found 30 missing values in `duration_min` and one missing value in `stations.capacity`. Since trip duration is essential for calculating average duration, I will exclude trips with missing durations from the analysis. The missing station capacity does not affect the analysis because capacity is not one of the variables required. I will retain the unusually long trips for now and flag them for investigation rather than assuming they are errors.

In [13]:
# Remove exact duplicate rows
trips_clean = trips.drop_duplicates().copy()

# Remove trips without a recorded duration
trips_clean = trips_clean.dropna(subset=["duration_min"]).copy()

print("Rows after cleaning:", len(trips_clean))
print("Missing durations remaining:", trips_clean["duration_min"].isna().sum())
print("Distinct trip IDs:", trips_clean["trip_id"].nunique())

Rows after cleaning: 570
Missing durations remaining: 0
Distinct trip IDs: 570


## 3. Join and aggregate

*Join the tables and calculate the answer. Note how many rows you expect after each join, then check.*

In [14]:
# Join trips with bikes to obtain bike_type
trips_with_bikes = trips_clean.merge(
    tables["bikes"],
    on="bike_id",
    how="left",
    indicator=True
)

print("Trips after joining bikes:", len(trips_with_bikes))
print("\nBike join results:")
print(trips_with_bikes["_merge"].value_counts())

# Remove the merge indicator before the next join
trips_with_bikes = trips_with_bikes.drop(columns="_merge")

Trips after joining bikes: 570

Bike join results:
_merge
both          570
left_only       0
right_only      0
Name: count, dtype: int64


I will join the cleaned trips table with the bikes table using `bike_id` to identify each bike's type. I will then join the stations table using `start_station_id` and `station_id` to identify the zone where each trip started. Finally, I will group the data by bike type and starting-station zone to calculate the number of trips and the average trip duration in minutes.

In [16]:
# Join with stations to obtain the starting station's zone
trips_joined = trips_with_bikes.merge(
    tables["stations"][["station_id", "zone"]],
    left_on="start_station_id",
    right_on="station_id",
    how="left",
    indicator=True
)

print("Rows after joining stations:", len(trips_joined))
print("\nStation join results:")
print(trips_joined["_merge"].value_counts())

print("\nMissing starting zones:", trips_joined["zone"].isna().sum())

Rows after joining stations: 570

Station join results:
_merge
both          570
left_only       0
right_only      0
Name: count, dtype: int64

Missing starting zones: 0


In [17]:
# Remove the merge indicator and extra station_id column
trips_joined = trips_joined.drop(columns=["_merge", "station_id"])

# Group by bike type and starting station zone
summary = (
    trips_joined
    .groupby(["bike_type", "zone"])
    .agg(
        trip_count=("trip_id", "count"),
        average_duration_min=("duration_min", "mean")
    )
    .reset_index()
)

# Round average duration to 2 decimal places
summary["average_duration_min"] = summary["average_duration_min"].round(2)

# Display the results
print(summary)

    bike_type    zone  trip_count  average_duration_min
0      E-bike  Centre           7                 13.29
1      E-bike   North           2                 21.00
2      E-bike   South           1                 21.00
3      E-bike    West           1                 15.00
4    Electric  Centre          50                 14.88
5    Electric    East          10                 15.50
6    Electric   North          15                 15.47
7    Electric   South          15                 13.60
8    Electric    West          19                 14.95
9   Electric   Centre          20                 14.65
10  Electric     East           1                 18.00
11  Electric    North           4                 15.25
12  Electric    South           4                 14.25
13  Electric     West           2                 23.00
14   STANDARD  Centre          32                 19.41
15   STANDARD    East           2                 14.50
16   STANDARD   North           4               

In [18]:
# Standardise bike type capitalisation and remove extra spaces
trips_joined["bike_type"] = (
    trips_joined["bike_type"]
    .str.strip()
    .str.lower()
    .str.capitalize()
)

# Check the standardised categories
print(trips_joined["bike_type"].value_counts())

bike_type
Standard    409
Electric    150
E-bike       11
Name: count, dtype: int64


In [19]:
summary = (
    trips_joined
    .groupby(["bike_type", "zone"])
    .agg(
        trip_count=("trip_id", "count"),
        average_duration_min=("duration_min", "mean")
    )
    .reset_index()
)

summary["average_duration_min"] = summary["average_duration_min"].round(2)

print(summary)

   bike_type    zone  trip_count  average_duration_min
0     E-bike  Centre           7                 13.29
1     E-bike   North           2                 21.00
2     E-bike   South           1                 21.00
3     E-bike    West           1                 15.00
4   Electric  Centre          74                 14.62
5   Electric    East          11                 15.73
6   Electric   North          21                 15.48
7   Electric   South          23                 13.09
8   Electric    West          21                 15.71
9   Standard  Centre         217                 38.66
10  Standard    East          20                 19.50
11  Standard   North          39                 20.36
12  Standard   South          60                 43.15
13  Standard    West          73                 19.82


## 4. Check

*How do you know the answer is right? Compare a count, spot check a few records, or recalculate one number a different way. What did you find?*

Both joins preserved all 570 cleaned trips. All trips matched a bike record and a starting station, and no starting zones were missing. The aggregation groups trips by bike type and starting-station zone and calculates the trip count and average duration. I also standardised inconsistent capitalisation in bike types so that the same category is not split across multiple rows.
I will check that the trip counts in the summary add up to 570 and that the summary contains no missing values in the grouping or result columns.

In [20]:
print("Cleaned trips:", len(trips_clean))
print("Trips counted in summary:", summary["trip_count"].sum())
print("Missing values in summary:")
print(summary.isnull().sum())
print("\nNumber of summary groups:", len(summary))

Cleaned trips: 570
Trips counted in summary: 570
Missing values in summary:
bike_type               0
zone                    0
trip_count              0
average_duration_min    0
dtype: int64

Number of summary groups: 14


In [21]:
print(
    "Duplicate bike type-zone groups:",
    summary.duplicated(subset=["bike_type", "zone"]).sum()
)

Duplicate bike type-zone groups: 0


## 5. Prompts and AI notes

*Paste each prompt as you sent it. Note what you accepted, changed or rejected and why.*

## 6. What I could not verify

*What remains unchecked? What would you do next?*